# Tech Challenge Fase 3 — Diagnóstico da Base de Modelagem

## Objetivo

Validar se os dados construídos na Fase 2 possuem qualidade, volume e estrutura suficientes para o desenvolvimento de um modelo supervisionado capaz de prever se um aluno será alfabetizado ou não alfabetizado.

Nesta etapa serão avaliados:
- volume de dados;
- distribuição temporal;
- balanceamento da variável-alvo;
- cobertura territorial;
- duplicidades aluno × ano;
- população candidata ao teste temporal;
- possíveis riscos de data leakage.


## 1. Visão geral da base

Verifica volume, quantidade de alunos, cobertura temporal e territorial e balanceamento do target `alfabetizado`.


In [ ]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_aluno) AS alunos_distintos,
    COUNT(DISTINCT ano) AS qtd_anos,
    MIN(ano) AS primeiro_ano,
    MAX(ano) AS ultimo_ano,
    COUNT(DISTINCT id_municipio) AS qtd_municipios,
    COUNT(DISTINCT sigla_uf) AS qtd_ufs,
    COUNT(DISTINCT rede) AS qtd_redes,
    COUNT(DISTINCT serie) AS qtd_series,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(100 * AVG(CASE WHEN alfabetizado = TRUE THEN 1.0 WHEN alfabetizado = FALSE THEN 0.0 END), 2) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos;


## 2. Distribuição temporal do target

Avalia a proporção de alfabetizados e não alfabetizados em 2023 e 2024.


In [ ]:
%sql

SELECT
    ano,
    COUNT(*) AS total_alunos,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(100 * AVG(CASE WHEN alfabetizado = TRUE THEN 1.0 WHEN alfabetizado = FALSE THEN 0.0 END), 2) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos
GROUP BY ano
ORDER BY ano;


## 3. Duplicidade aluno × ano

Um mesmo aluno pode aparecer em anos diferentes. O problema seria encontrar mais de um registro para a mesma combinação `id_aluno + ano`.


In [ ]:
%sql

SELECT COUNT(*) AS combinacoes_duplicadas
FROM (
    SELECT ano, id_aluno, COUNT(*) AS quantidade
    FROM workspace.alfabetizacao_silver.fato_alunos
    GROUP BY ano, id_aluno
    HAVING COUNT(*) > 1
);


## 4. Alunos presentes nos dois anos

Mede quantos alunos aparecem em 2023 e 2024 para orientar a separação entre treino e teste.


In [ ]:
%sql

SELECT COUNT(*) AS alunos_presentes_em_2023_e_2024
FROM (
    SELECT id_aluno
    FROM workspace.alfabetizacao_silver.fato_alunos
    GROUP BY id_aluno
    HAVING COUNT(DISTINCT ano) > 1
);


## 5. População candidata ao teste final

O teste final será formado por alunos de 2024 que não aparecem em 2023.


In [ ]:
%sql

SELECT
    COUNT(*) AS alunos_teste_2024_novos,
    SUM(CASE WHEN alfabetizado = TRUE THEN 1 ELSE 0 END) AS alfabetizados,
    SUM(CASE WHEN alfabetizado = FALSE THEN 1 ELSE 0 END) AS nao_alfabetizados,
    ROUND(100 * AVG(CASE WHEN alfabetizado = TRUE THEN 1.0 WHEN alfabetizado = FALSE THEN 0.0 END), 2) AS percentual_alfabetizados
FROM workspace.alfabetizacao_silver.fato_alunos a
WHERE a.ano = 2024
  AND NOT EXISTS (
      SELECT 1
      FROM workspace.alfabetizacao_silver.fato_alunos b
      WHERE b.ano = 2023
        AND b.id_aluno = a.id_aluno
  );


## 6. Valores ausentes

Verifica a completude das principais variáveis candidatas ao modelo.


In [ ]:
%sql

SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN alfabetizado IS NULL THEN 1 ELSE 0 END) AS target_nulo,
    SUM(CASE WHEN id_municipio IS NULL THEN 1 ELSE 0 END) AS municipio_nulo,
    SUM(CASE WHEN sigla_uf IS NULL THEN 1 ELSE 0 END) AS uf_nula,
    SUM(CASE WHEN rede IS NULL THEN 1 ELSE 0 END) AS rede_nula,
    SUM(CASE WHEN serie IS NULL THEN 1 ELSE 0 END) AS serie_nula,
    SUM(CASE WHEN presenca IS NULL THEN 1 ELSE 0 END) AS presenca_nula
FROM workspace.alfabetizacao_silver.fato_alunos;


## 7. Diagnóstico inicial de data leakage

A variável `proficiencia` deve ser analisada com cuidado porque a classificação de alfabetização está diretamente associada ao desempenho na avaliação.


In [ ]:
%sql

SELECT
    alfabetizado,
    COUNT(*) AS quantidade,
    ROUND(AVG(proficiencia), 2) AS media_proficiencia,
    ROUND(MIN(proficiencia), 2) AS menor_proficiencia,
    ROUND(MAX(proficiencia), 2) AS maior_proficiencia,
    ROUND(percentile_approx(proficiencia, 0.50), 2) AS mediana_proficiencia
FROM workspace.alfabetizacao_silver.fato_alunos
WHERE alfabetizado IS NOT NULL
  AND proficiencia IS NOT NULL
GROUP BY alfabetizado
ORDER BY alfabetizado;


%md
## Conclusão do diagnóstico

O diagnóstico confirmou que a base possui condições adequadas para o desenvolvimento do modelo supervisionado.

### Principais conclusões

- **Volume suficiente:** 3.867.999 registros e 2.352.328 alunos distintos.
- **Target balanceado:** 51,31% dos registros estão classificados como alfabetizados e 48,69% como não alfabetizados.
- **Cobertura temporal:** dados disponíveis para 2023 e 2024.
- **Cobertura territorial ampla:** 5.548 municípios e 26 UFs representadas.
- **Qualidade das chaves:** não foram encontrados valores nulos nas principais variáveis utilizadas no diagnóstico.
- **Sem duplicidade por aluno e ano:** nenhuma combinação `id_aluno + ano` duplicada foi identificada.
- **Teste temporal viável:** 604.889 alunos de 2024 não aparecem em 2023 e poderão ser utilizados como conjunto de teste final.
- **Distribuição do teste balanceada:** 49,28% alfabetizados e 50,72% não alfabetizados.
- **Data leakage identificado:** a variável `proficiencia` apresenta relação praticamente determinística com o target `alfabetizado` e, portanto, será excluída das features do modelo.
- **Variável sem poder discriminatório:** `serie` possui apenas uma categoria e também não deverá contribuir para a modelagem.

### Decisão metodológica

Os dados de **2023 serão utilizados para desenvolvimento, treinamento e validação cruzada**, enquanto os **604.889 alunos novos de 2024 serão reservados para o teste final fora da amostra**.

O próximo passo será enriquecer a base com variáveis socioeconômicas e construir a tabela Gold `base_modelagem_aluno`, que será utilizada nas etapas de análise exploratória e Machine Learning.